# TP 8 : Les Données Problématiques (Bad Data)
(SUJET GUIDÉ)

**Dataset :** Diabetes 130-US Hospitals (UCI ML Repository)  
**Contexte :** 101 766 hospitalisations réelles dans 130 hôpitaux américains (1999-2008).  
**Tâche :** Prédire la réadmission hospitalière à 30 jours.

---

> **"Garbage in, garbage out"** — Un modèle entraîné sur des données sales produira des prédictions incorrectes, même avec l'algorithme le plus sophistiqué.

## Ce que ce dataset contient comme problèmes *réels* :

| # | Problème | Exemple concret |
|---|---|---|
| 1 | **Valeurs manquantes encodées en texte** | `'?'` au lieu de `NaN` dans 3 colonnes importantes |
| 2 | **Taux de manquants extrêmement élevé** | `weight` : 97% manquant — inutilisable |
| 3 | **Déséquilibre des classes** | ~11% de réadmissions <30j vs ~89% non-réadmissions |
| 4 | **Patients en doublon** | 101k séjours pour 71k patients — même patient plusieurs fois |
| 5 | **Variables à haute cardinalité** | `diag_1` : 700+ codes ICD-9 → impossible à one-hot encoder |

In [ ]:
# !pip install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.impute import SimpleImputer

np.random.seed(42)
pd.set_option('display.max_columns', 20)

---
## Chargement des Données Brutes

In [ ]:
print("Chargement du dataset diabetes-130...")
dataset = fetch_ucirepo(id=296)
# encounter_id et patient_nbr sont dans dataset.data.ids (pas dans features)
df_raw = pd.concat([dataset.data.ids, dataset.data.features, dataset.data.targets], axis=1)

print(f"Dimensions : {df_raw.shape}")
print(f"\nTypes de données :")
print(df_raw.dtypes.value_counts())
df_raw.head(3)

---
## Problème 1 : Valeurs Manquantes Encodées en Texte (`'?'`)

Dans ce dataset, les valeurs manquantes ne sont **pas** des `NaN` Python — elles sont encodées comme la chaîne de caractères `'?'`.
Si vous ne les convertissez pas, scikit-learn plantera ou, pire, traitera `'?'` comme une **catégorie valide**.

Ce type d'erreur est très fréquent avec des données CSV venant de systèmes hospitaliers ou administratifs.

In [ ]:
# Observons le problème
print("Valeurs uniques dans 'race' :")
print(df_raw['race'].value_counts(dropna=False))

print("\nValeurs uniques dans 'weight' (20 premières) :")
print(df_raw['weight'].value_counts(dropna=False).head(20))

print("\nValeurs uniques dans 'payer_code' (5 premières) :")
print(df_raw['payer_code'].value_counts(dropna=False).head(5))

In [ ]:
# TODO : Comptez le nombre de '?' par colonne
# Indice : pour chaque colonne, compter les lignes où la valeur == '?'

# nb_interrogation = (df_raw == '?').sum()
# nb_interrogation = nb_interrogation[nb_interrogation > 0].sort_values(ascending=False)
# print("Colonnes avec des '?' :")
# print(nb_interrogation)

# TODO : Calculez le pourcentage de '?' pour chaque colonne concernée

In [ ]:
# TODO : Corrigez en remplaçant '?' par NaN
# df = df_raw.copy()
# df.replace('?', np.nan, inplace=True)

# Vérifiez : comptez maintenant les NaN vrais
# print(df.isnull().sum()[df.isnull().sum() > 0])

---
## Problème 2 : Colonnes avec Trop de Valeurs Manquantes

Une fois les `'?'` convertis en `NaN`, certaines colonnes ont un taux de manquants si élevé qu'elles deviennent inutilisables.

**Règle générale :**
- < 5% manquant → imputation par médiane/mode acceptable
- 5–30% → imputation avec prudence ou encoder l'absence comme feature
- > 30% → envisager de **supprimer la colonne** (le bruit dépasse le signal)
- > 70% → **supprimer la colonne** (quasi-inutile)

In [ ]:
# (Exécutez ce bloc après avoir créé df avec les '?' → NaN)
# df = df_raw.copy()
# df.replace('?', np.nan, inplace=True)

# TODO : Calculez et visualisez le taux de NaN par colonne

# nan_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
# nan_pct_nonzero = nan_pct[nan_pct > 0]
# print(nan_pct_nonzero.round(1))

# TODO : Affichez un barplot horizontal des taux de NaN
# Marquez des lignes horizontales à 30% et 70%
# Interprétez : quelles colonnes faut-il supprimer ?

In [ ]:
# TODO : Décidez quelles colonnes supprimer
# Proposition : supprimer les colonnes avec > 40% de NaN

# SEUIL = 40  # %
# cols_a_supprimer = nan_pct[nan_pct > SEUIL].index.tolist()
# print(f"Colonnes supprimées (>{SEUIL}% NaN) : {cols_a_supprimer}")

# df_clean = df.drop(columns=cols_a_supprimer)
# print(f"Dimensions après suppression : {df_clean.shape}")

---
## Problème 3 : Déséquilibre des Classes

La variable cible `readmitted` a 3 valeurs : `<30`, `>30`, `No`.  
Si l'on binarise en "réadmis dans les 30 jours" (1) vs "non" (0), le déséquilibre est sévère.

Un modèle naïf peut atteindre ~89% d'accuracy en prédisant **toujours** la classe majoritaire... sans rien apprendre.

In [ ]:
# Analyse de la variable cible
# (Supposons df créé et '?' remplacés)

# TODO : Affichez la distribution de 'readmitted' (3 classes)
# Puis la distribution binaire readmit_30

# df['readmit_30'] = (df['readmitted'] == '<30').astype(int)

# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# df['readmitted'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', rot=0)
# axes[0].set_title('Distribution originale (3 classes)')
# df['readmit_30'].value_counts().plot(kind='bar', ax=axes[1],
#                                       color=['#3498db', '#e74c3c'], rot=0)
# axes[1].set_title('Cible binaire : réadmis <30j')
# plt.tight_layout()
# plt.show()

# Quelle est la baseline (accuracy du modèle "toujours classe majoritaire") ?

In [ ]:
# Construction du dataset de travail minimal
# (features numériques pour simplifier)

# df['readmit_30'] = (df['readmitted'] == '<30').astype(int)
# FEATS_NUM = [
#     'time_in_hospital', 'num_lab_procedures', 'num_procedures',
#     'num_medications', 'number_outpatient', 'number_emergency',
#     'number_inpatient', 'number_diagnoses'
# ]
# df_work = df[FEATS_NUM + ['readmit_30']].dropna()
# X = df_work[FEATS_NUM].values
# y = df_work['readmit_30'].values

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

# TODO : Entraînez un modèle NAÏF (LogisticRegression, sans class_weight)
# Affichez : accuracy ET classification_report
# Observez le Recall de la classe 1 (réadmis <30j)
# Que se passe-t-il ?

In [ ]:
# TODO : Corrigez avec class_weight='balanced'
# Comparez les deux modèles avec des matrices de confusion côte à côte
# Quel modèle détecte mieux les patients à risque de réadmission ?

# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# ConfusionMatrixDisplay.from_predictions(..., ax=axes[0])
# ConfusionMatrixDisplay.from_predictions(..., ax=axes[1])
# plt.show()

---
## Problème 4 : Patients en Doublon (Multiple Encounters)

Ce dataset contient **plusieurs séjours** pour certains patients (même `patient_nbr`).  
Si on garde tous les séjours :
- Le même patient peut être dans train et test → **fuite** (cf. TP7)
- Les données sont corrélées → les hypothèses d'indépendance des observations sont violées
- Les patients avec beaucoup d'hospitalisations sont **sur-représentés**

**Stratégie courante :** ne garder que le **premier séjour** de chaque patient.

In [ ]:
# TODO : Analysez les patients multi-séjours

# 1. Combien de patients uniques y a-t-il ? (patient_nbr)
# print(f"Séjours totaux    : {len(df)}")
# print(f"Patients uniques  : {df['patient_nbr'].nunique()}")
# print(f"Séjours en trop   : {len(df) - df['patient_nbr'].nunique()}")

# 2. Distribution du nombre de séjours par patient
# sejours_par_patient = df.groupby('patient_nbr').size().value_counts().sort_index()
# print(sejours_par_patient.head(10))

In [ ]:
# TODO : Conservez uniquement le premier séjour de chaque patient
# Indice : les séjours sont ordonnés par encounter_id (croissant = plus ancien)

# df_dedup = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first')
# print(f"Après dédoublonnage : {len(df_dedup)} lignes")

# Question : le taux de réadmission change-t-il après dédoublonnage ?
# print(f"Taux réadmission avant : {df['readmit_30'].mean():.2%}")
# print(f"Taux réadmission après : {df_dedup['readmit_30'].mean():.2%}")

---
## Problème 5 : Variables à Haute Cardinalité

Les diagnostics (`diag_1`, `diag_2`, `diag_3`) sont des codes **ICD-9** avec ~700 valeurs possibles chacun.  
La colonne `medical_specialty` a 84 valeurs distinctes.

**Le problème :** un `pd.get_dummies()` naïf crée des milliers de colonnes → explosion mémoire + **overfitting**.

**Solution :** regrouper les modalités rares sous `'Autre'` (ne conserver que les N plus fréquentes).

In [ ]:
# TODO : Analysez la cardinalité

# cols_cat = ['race', 'gender', 'age', 'admission_type_id',
#             'discharge_disposition_id', 'diag_1', 'medical_specialty']

# for col in cols_cat:
#     n_unique = df_raw[col].replace('?', np.nan).nunique(dropna=True)
#     print(f"{col:35s} : {n_unique:5d} valeurs distinctes")

In [ ]:
# TODO : Implémentez une fonction pour regrouper les modalités rares

# def grouper_rares(serie, top_n=10, label_autre='Autre'):
#     """Remplace les modalités hors du top_n par label_autre."""
#     top_categories = serie.value_counts().nlargest(top_n).index
#     return serie.where(serie.isin(top_categories), other=label_autre)

# Testez sur diag_1 :
# print(f"Avant : {df['diag_1'].nunique()} modalités")
# diag1_reduit = grouper_rares(df['diag_1'], top_n=15)
# print(f"Après (top 15) : {diag1_reduit.nunique()} modalités")
# print(diag1_reduit.value_counts())

---
## Exercice Final : Pipeline de Nettoyage Complet

Comparez les performances d'un Random Forest sur **données brutes** vs **données nettoyées**.

Implémentez la fonction `nettoyer_diabetes()` ci-dessous, puis évaluez les deux modèles.

In [ ]:
def grouper_rares(serie, top_n=10, label_autre='Autre'):
    top_categories = serie.value_counts().nlargest(top_n).index
    return serie.where(serie.isin(top_categories), other=label_autre)


def nettoyer_diabetes(df_input):
    """
    Pipeline de nettoyage complet pour diabetes-130.
    Retourne X (array numpy), y (array numpy).
    """
    df = df_input.copy()

    # TODO : Étape 1 — Remplacer '?' par NaN

    # TODO : Étape 2 — Créer la cible binaire readmit_30

    # TODO : Étape 3 — Supprimer les colonnes avec trop de NaN (seuil : 40%)

    # TODO : Étape 4 — Dédoublonner (garder premier séjour par patient)

    # TODO : Étape 5 — Regrouper les modalités rares de diag_1 (top 20)

    # TODO : Étape 6 — Encoder les variables catégorielles (LabelEncoder ou get_dummies)

    # TODO : Étape 7 — Imputer les NaN restants par la médiane (features numériques)

    # TODO : Retourner X, y
    # features = [c for c in df.columns if c not in ['readmit_30', 'readmitted', 'patient_nbr', 'encounter_id']]
    # X = df[features].values
    # y = df['readmit_30'].values
    # return X, y
    pass

In [ ]:
# TODO : Évaluez et comparez

# Données nettoyées
# X_clean, y_clean = nettoyer_diabetes(df_raw)
# print(f"Dataset nettoyé : {X_clean.shape}")

# Évaluez un Random Forest (class_weight='balanced', n_estimators=100)
# sur les données nettoyées avec train_test_split (stratify=y_clean)

# Comparez :
#   - Accuracy
#   - F1-score (classe 1 = réadmis <30j)
#   - Recall (classe 1) — critique en médecine

# Conclusion : le nettoyage améliore-t-il les performances ?

---
## Récapitulatif : Checklist Qualité des Données

| Problème | Détection | Traitement |
|---|---|---|
| Manquants encodés en texte | `df.isin(['?']).sum()` | `.replace('?', np.nan)` |
| Trop de NaN dans une colonne | `df.isnull().mean()` | Supprimer si > 40-70% |
| Déséquilibre des classes | `y.value_counts()` | `class_weight='balanced'`, F1 plutôt qu'accuracy |
| Patients en doublon | `df['patient_nbr'].nunique()` vs `len(df)` | `drop_duplicates(subset='patient_nbr')` |
| Haute cardinalité | `df[col].nunique()` | Regrouper les modalités rares |

### Questions de réflexion
1. Pourquoi encoder les manquants comme `'?'` au lieu de `NaN` est-il dangereux pour un modèle de ML ?
2. La colonne `weight` a 97% de NaN. Dans quels cas vaut-il quand même la peine de la garder ?
3. Sur un dataset médical déséquilibré, pourquoi préférer le **Recall** de la classe positive à l'**Accuracy** ?
4. Quels risques y a-t-il à regrouper les codes ICD-9 rares sous `'Autre'` ? Proposez une alternative.